[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/32_topk_sampling.ipynb)

# 🟠 Medium: Top-k / Top-p (Nucleus) Sampling

Implement **sampling with top-k and top-p filtering** — the standard LLM decoding strategy.

### Signature
```python
def sample_top_k_top_p(logits, top_k=0, top_p=1.0, temperature=1.0) -> int:
    # logits: (V,) unnormalized log-probabilities
    # Returns: sampled token index
```

### Algorithm
1. Scale by temperature: `logits /= temperature`
2. Top-k: keep only top-k logits, set rest to `-inf`
3. Top-p: sort by prob, mask tokens where cumulative prob exceeds p
4. Sample from filtered distribution

In [1]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 1.3 MB/s eta 0:00:00


In [2]:
import torch

In [87]:
# ✏️ YOUR IMPLEMENTATION HERE

def sample_top_k_top_p(logits, top_k=0, top_p=1.0, temperature=1.0):
    p = torch.softmax(logits / temperature, dim=-1)
    n = len(logits)
    sorted_p, sorted_indices = torch.sort(p, descending=True)
    if top_k:
      sorted_indices = sorted_indices[:top_k]
      sorted_p = sorted_p[:top_k]

    if top_p < 1:
      presum = torch.cumsum(sorted_p, dim=-1)
      keep = presum <= top_p
      keep[0] = True
      sorted_indices =  sorted_indices[keep]
      sorted_p =  sorted_p[keep]

    probs = sorted_p / torch.sum(sorted_p)

    sampled_idx = torch.multinomial(probs, num_samples=1)
    next_token = sorted_indices[sampled_idx].item()
    return next_token

    # pass  # temperature, top-k filter, top-p filter, sample

In [88]:
# 🧪 Debug
logits = torch.tensor([1.0, 5.0, 2.0, 0.5])
print('top_k=1:', sample_top_k_top_p(logits.clone(), top_k=1))
print('top_p=0.5:', sample_top_k_top_p(logits.clone(), top_p=0.5))
print('temp=0.01:', sample_top_k_top_p(logits.clone(), temperature=0.01))

top_k=1: 1
top_p=0.5: 1
temp=0.01: 1


In [89]:
# ✅ SUBMIT
from torch_judge import check
check('topk_sampling')


🧪 Testing: Top-k / Top-p Sampling (Medium)
──────────────────────────────────────────────────
  ✅ [1/4] top_k=1 always returns argmax (7.5ms)
  ✅ [2/4] Low temperature concentrates (12.4ms)
  ✅ [3/4] All tokens reachable (no filtering) (354.9ms)
  ✅ [4/4] Returns valid index (6.9ms)
──────────────────────────────────────────────────
  🎉 All 4 tests passed! (381.7ms total)
  Progress saved. Run status() to see your dashboard.

